In [3]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [24]:
torch.backends.cudnn.benchmark = True

In [12]:
TRAIN_DIR = "data/Processed_Images/Processed_Train/"
VALID_DIR = "data/Processed_Images/Processed_valid/"
TEST_DIR = "data/Processed_Images/Processed_test/"

TRAIN_CSV = "data/Processed_Images/Processed_Train/_annotations.csv"
VALID_CSV = "data/Processed_Images/Processed_valid/_annotations.csv"
TEST_CSV = "data/Processed_Images/Processed_test/_annotations.csv"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3 # 0.001
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

In [6]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


In [7]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

])

In [8]:
valid_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

])

In [13]:
class SnakeDataset(Dataset):

    def __init__(self, csv_path, image_folder, transform=None):
        """
        Args:
            csv_path (str): Path to processed_annotations.csv
            image_folder (str): Folder containing cropped images
            transform (callable): torchvision transforms
        """

        self.annotations = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_folder,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["Label"])

        return image, label

    @property
    def classes(self):
        return sorted(
            self.annotations["Class"].unique().tolist()
        )

    @property
    def num_classes(self):
        return self.annotations["Label"].nunique()

In [15]:
train_dataset = SnakeDataset(
    csv_path=TRAIN_CSV,
    image_folder=TRAIN_DIR,
    transform=train_transform
)

valid_dataset = SnakeDataset(
    csv_path=VALID_CSV,
    image_folder=VALID_DIR,
    transform=valid_transform
)

test_dataset = SnakeDataset(
    csv_path=TEST_CSV,
    image_folder=TEST_DIR,
    transform=valid_transform
)

In [16]:
print(f"Train Images      : {len(train_dataset)}")
print(f"Validation Images : {len(valid_dataset)}")
print(f"Test Images       : {len(test_dataset)}")

print(f"Classes           : {train_dataset.num_classes}")

Train Images      : 6109
Validation Images : 572
Test Images       : 290
Classes           : 15


In [17]:
print("Total Images:", len(train_dataset))

Total Images: 6109


In [23]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,          # Increase if VRAM allows
    shuffle=True,
    num_workers=6,          # Start here
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=6,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

In [20]:
print(len(train_dataset))
print(len(valid_dataset))

6109
572


In [22]:
import platform
print(platform.processor())

Intel64 Family 6 Model 141 Stepping 1, GenuineIntel


In [25]:
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1

model = convnext_tiny(weights=weights)

num_features = model.classifier[2].in_features

model.classifier[2] = nn.Linear(
    num_features,
    15
)

model = model.to(DEVICE)

print(model.classifier)

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\kisha/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:11<00:00, 10.3MB/s] 


Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=15, bias=True)
)


In [26]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

In [27]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [28]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [29]:
scaler = torch.amp.GradScaler("cuda")

In [30]:
print("Model Loaded Successfully")
print(f"Classes : 15")
print(f"Device  : {DEVICE}")
print(f"Optimizer : {optimizer.__class__.__name__}")
print(f"Scheduler : {scheduler.__class__.__name__}")

Model Loaded Successfully
Classes : 15
Device  : cuda
Optimizer : AdamW
Scheduler : CosineAnnealingLR


In [31]:
from tqdm.auto import tqdm
import torch


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, labels in progress_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

        progress_bar.set_postfix({
            "Loss": f"{running_loss/(progress_bar.n+1):.4f}",
            "Acc": f"{100*correct/total:.2f}%"
        })

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [32]:
@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Validation", leave=False):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [33]:
import os

os.makedirs("checkpoints", exist_ok=True)


def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    best_accuracy,
    filename="checkpoints/best_model.pth"
):

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_accuracy": best_accuracy
    }, filename)